# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

First, let's list all record sets with their `@id` and name. Then, for a chosen record set, we will print all its fields and available columns, referencing their `@id`.

In [ ]:
# List all record sets in the metadata and show their @ids and names
print("Available record sets (@id and name):")
record_sets = []
for rs in metadata.record_sets:
    print(f"- @id: {rs.id} | name: {getattr(rs, 'name', '(no name)')}")
    record_sets.append(rs.id)

if len(record_sets) > 0:
    example_record_set_id = record_sets[0]
    # Get the record set object
    record_set_obj = [rs for rs in metadata.record_sets if rs.id == example_record_set_id][0]
    print(f"\nFields in record set '@id': {example_record_set_id}")
    for field in record_set_obj.fields:
        print(f"  - Field @id: {field.id} | name: {getattr(field, 'name', '(no name)')} | type: {getattr(field, 'data_type', '-')}")
    # List columns for completeness if present
    if hasattr(record_set_obj, 'columns'):
        print(f"\nColumns in record set '@id': {example_record_set_id}")
        for col in record_set_obj.columns:
            print(f"  - Column @id: {col.id} | name: {getattr(col, 'name', '(no name)')}")

## 3. Data Extraction
Load data from all record sets into DataFrames for analysis. We will use the record set and field `@id`s from the overview.

First, let's extract each record set to a pandas DataFrame, and display the columns for one of them as an example.

In [ ]:
# Extract data from each record set by their @id
all_record_set_ids = record_sets
dataframes = {}

for record_set_id in all_record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Show columns for the first record set
if all_record_set_ids:
    first_id = all_record_set_ids[0]
    print(f"Columns in DataFrame for record set '@id': {first_id}")
    print(dataframes[first_id].columns.tolist())
    display(dataframes[first_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Here, we select a numeric field from a record set (using its `@id`), filter the DataFrame, normalize the field, and group by a key attribute for summary statistics.

_Reminder: All record sets, fields and columns must be referenced by their `@id`._

In [ ]:
# Choose the first available record set as example
record_set_id = all_record_set_ids[0]
df = dataframes[record_set_id]

# Show available numeric fields (infer by dtype for demo)
print(f"Numeric fields in record set '@id': {record_set_id}")
numeric_fields = df.select_dtypes(include='number').columns.tolist()
print(numeric_fields)
# If none found, skip further steps

if numeric_fields:
    numeric_field_id = numeric_fields[0]
    # Set threshold as an example (use a default if uncertain)
    threshold = df[numeric_field_id].quantile(0.5) if not df[numeric_field_id].empty else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records in '@id': {record_set_id} where '{numeric_field_id}' > {threshold}")
    display(filtered_df.head())

    # Normalize the numeric field in the filtered DataFrame
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Find a likely groupable field (e.g., categorical string type)
    group_field = None
    for col in df.columns:
        if df[col].dtype == object and col != numeric_field_id:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean '{numeric_field_id}' by '{group_field}' for '@id': {record_set_id}")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we plot the numeric field distribution, and if grouping is possible, a barplot of means per group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_fields:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of '{numeric_field_id}' in record set '@id': {record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    
    # Barplot if group field exists
    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(8,4))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field_id)
        plt.title(f"Mean '{numeric_field_id}' by '{group_field}' in '@id': {record_set_id}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² dataset provides richly structured, tabular records of cancer survivors with second primary colorectal cancer, annotated for clinicopathological and molecular variables.
- Through this notebook, we demonstrated programmatic discovery of record sets and fields via their `@id`s using the `mlcroissant` API, robust data loading, and core exploratory analysis in pandas.
- The example EDA illustrated how to filter on numeric fields, normalize data, and group for quick summaries—practices essential for advanced biomedical/clinical data science workflows.
- Visual inspection supports initial discovery and may inform hypothesis generation or model development in further research with this dataset.